# Web Research Agent ReAct Alt


In [ ]:
# ============================================================================
# Web Research Agent using LangGraph and ReAct Pattern
# ============================================================================
# This implementation creates a ReAct (Reasoning + Acting) agent that:
# 1. Plans research by generating key questions using an LLM
# 2. Acts by searching the web for answers using Tavily
# 3. Compiles a structured research report
# ============================================================================

from typing import TypedDict, Annotated, List, Dict
import operator
from langgraph.graph import StateGraph, END
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from tavily import TavilyClient
import os
from datetime import datetime
from dotenv import load_dotenv

load_dotenv()

# Helper to merge dict state safely for LangGraph

def merge_search_results(old: Dict[str, List[Dict]] | None, new: Dict[str, List[Dict]] | None) -> Dict[str, List[Dict]]:
    if old is None:
        return new or {}
    if new is None:
        return old or {}
    merged = dict(old)
    merged.update(new)
    return merged

# ============================================================================
# CONFIGURATION
# ============================================================================
# Set your API keys via environment variables (loaded above). Do not hardcode secrets.
# Example: export OPENAI_API_KEY=... and TAVILY_API_KEY=...

# ============================================================================
# STATE DEFINITION
# ============================================================================

class AgentState(TypedDict):
    """
    Defines the state of our ReAct agent.
    
    Attributes:
        topic: The research topic provided by the user
        research_questions: List of questions generated by the LLM
        search_results: Dictionary mapping questions to their search results
        final_report: The compiled research report
        current_step: Tracks the current phase (planning/acting/reporting)
        messages: List of messages for tracking agent reasoning
    """
    topic: str
    research_questions: List[str]
    search_results: Annotated[Dict[str, List[Dict]], merge_search_results]
    final_report: str
    current_step: str
    messages: Annotated[List, operator.add]


# ============================================================================
# AGENT CLASS
# ============================================================================

class WebResearchAgent:
    """
    A ReAct agent that performs web research on a given topic.
    
    The agent follows three phases:
    1. Planning: Generate research questions using LLM
    2. Acting: Search the web for answers to each question
    3. Reporting: Compile findings into a structured report
    """
    
    def __init__(self, 
                 model_name: str = "gpt-4o-mini",
                 temperature: float = 0.7,
                 num_questions: int = 5,
                 max_search_results: int = 5):
        """
        Initialize the Web Research Agent.
        
        Args:
            model_name: The LLM model to use (default: gpt-4o-mini)
            temperature: LLM temperature for creativity (default: 0.7)
            num_questions: Number of research questions to generate (default: 5)
            max_search_results: Maximum search results per question (default: 5)
        """
        self.llm = ChatOpenAI(model=model_name, temperature=temperature)
        self.tavily_client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))
        self.num_questions = num_questions
        self.max_search_results = max_search_results
        self.graph = self._build_graph()
    
    # ========================================================================
    # PLANNING PHASE - Generate Research Questions
    # ========================================================================
    
    def planning_node(self, state: AgentState) -> AgentState:
        """
        Planning Phase: Use LLM to generate research questions.
        
        This node implements the "Reasoning" part of ReAct pattern.
        The LLM analyzes the topic and generates structured research questions
        that cover different aspects of the topic.
        
        Args:
            state: Current agent state containing the research topic
            
        Returns:
            Updated state with research questions
        """
        topic = state["topic"]
        
        # Construct a detailed prompt for the LLM to generate research questions
        system_prompt = """You are an expert research assistant. Your task is to generate 
        comprehensive research questions for a given topic. Generate questions that:
        1. Cover different aspects of the topic (causes, effects, solutions, trends)
        2. Are specific and answerable through web search
        3. Progress from fundamental to advanced understanding
        4. Include current developments and future implications
        
        Generate EXACTLY {num_questions} well-structured research questions.""".format(
            num_questions=self.num_questions
        )
        
        user_prompt = f"""Topic: {topic}

Please generate {self.num_questions} research questions about this topic. 
Format your response as a numbered list with ONLY the questions, one per line.
Example format:
1. What is...?
2. How does...?
3. Why is...?"""

        # Call the LLM to generate questions
        messages = [
            SystemMessage(content=system_prompt),
            HumanMessage(content=user_prompt)
        ]
        
        response = self.llm.invoke(messages)
        
        # Parse the response to extract questions
        questions_text = response.content
        questions = self._parse_questions(questions_text)
        
        # Update state
        state["research_questions"] = questions
        state["current_step"] = "planning_complete"
        state["messages"].append({
            "role": "planning",
            "content": f"Generated {len(questions)} research questions",
            "questions": questions
        })
        
        print(f"\n{'='*80}")
        print(f"🧠 PLANNING PHASE COMPLETE")
        print(f"{'='*80}")
        print(f"Generated {len(questions)} research questions for topic: '{topic}'")
        for i, q in enumerate(questions, 1):
            print(f"  {i}. {q}")
        print(f"{'='*80}\n")
        
        return state
    
    def _parse_questions(self, text: str) -> List[str]:
        """
        Parse research questions from LLM response.
        
        Args:
            text: Raw text from LLM containing questions
            
        Returns:
            List of cleaned research questions
        """
        lines = text.strip().split('\n')
        questions = []
        
        for line in lines:
            line = line.strip()
            if not line:
                continue
            
            # Remove numbering (1., 2., etc.) and extra whitespace
            # Handle various formats: "1.", "1)", "Question 1:", etc.
            import re
            cleaned = re.sub(r'^[\d]+[\.\)\:]?\s*', '', line)
            cleaned = re.sub(r'^[Qq]uestion\s+[\d]+[\.\)\:]?\s*', '', cleaned, flags=re.IGNORECASE)
            
            if cleaned and len(cleaned) > 10:  # Filter out very short lines
                questions.append(cleaned)
        
        # Ensure we have the right number of questions
        return questions[:self.num_questions]
    
    # ========================================================================
    # ACTING PHASE - Web Search
    # ========================================================================
    
    def acting_node(self, state: AgentState) -> AgentState:
        """
        Acting Phase: Search the web for answers to research questions.
        
        This node implements the "Acting" part of ReAct pattern.
        For each research question, the agent uses Tavily to search the web
        and extract relevant information.
        
        Args:
            state: Current agent state with research questions
            
        Returns:
            Updated state with search results
        """
        questions = state["research_questions"]
        search_results = {}
        
        print(f"\n{'='*80}")
        print(f"🔍 ACTING PHASE - WEB SEARCH")
        print(f"{'='*80}")
        
        for i, question in enumerate(questions, 1):
            print(f"\nSearching for Question {i}/{len(questions)}:")
            print(f"  '{question}'")
            
            try:
                # Perform web search using Tavily
                response = self.tavily_client.search(
                    query=question,
                    max_results=self.max_search_results,
                    search_depth="advanced"  # Use advanced search for better results
                )
                
                # Extract relevant information from search results
                results = []
                for result in response.get('results', []):
                    results.append({
                        'title': result.get('title', 'No title'),
                        'url': result.get('url', ''),
                        'content': result.get('content', 'No content available'),
                        'score': result.get('score', 0)
                    })
                
                search_results[question] = results
                print(f"  ✓ Found {len(results)} relevant sources")
                
            except Exception as e:
                print(f"  ✗ Error searching for question: {e}")
                search_results[question] = []
        
        # Update state
        state["search_results"] = search_results
        state["current_step"] = "acting_complete"
        state["messages"].append({
            "role": "acting",
            "content": f"Completed web search for {len(questions)} questions",
            "total_results": sum(len(v) for v in search_results.values())
        })
        
        print(f"\n{'='*80}")
        print(f"✓ ACTING PHASE COMPLETE")
        print(f"Total sources gathered: {sum(len(v) for v in search_results.values())}")
        print(f"{'='*80}\n")
        
        return state
    
    # ========================================================================
    # REPORTING PHASE - Compile Final Report
    # ========================================================================
    
    def reporting_node(self, state: AgentState) -> AgentState:
        """
        Reporting Phase: Compile research findings into a structured report.
        
        This node synthesizes all gathered information into a comprehensive
        report with proper structure and citations.
        
        Args:
            state: Current agent state with search results
            
        Returns:
            Updated state with final report
        """
        topic = state["topic"]
        questions = state["research_questions"]
        search_results = state["search_results"]
        
        print(f"\n{'='*80}")
        print(f"📝 REPORTING PHASE - COMPILING REPORT")
        print(f"{'='*80}\n")
        
        # Use LLM to synthesize the report
        system_prompt = """You are an expert research writer. Your task is to compile 
        a comprehensive, well-structured research report based on gathered information.
        
        Guidelines:
        1. Write in a clear, professional academic style
        2. Synthesize information from multiple sources
        3. Include relevant citations with [Source](URL) format
        4. Organize information logically
        5. Provide insightful analysis and connections between findings
        6. Write a strong introduction and conclusion
        """
        
        # Prepare the research data for the LLM
        research_data = self._format_research_data(questions, search_results)
        
        user_prompt = f"""Topic: {topic}

Research Data:
{research_data}

Please compile a comprehensive research report with the following structure:

# {topic} - Research Report

## Introduction
[Provide an engaging introduction to the topic]

## Research Findings

[For each research question, create a detailed section that:
- Answers the question comprehensively
- Synthesizes information from multiple sources
- Includes citations in [Source Title](URL) format
- Provides analysis and insights]

## Conclusion
[Summarize key findings and provide insights about the topic's significance]

## References
[List all sources used]

Write the complete report now:"""

        messages = [
            SystemMessage(content=system_prompt),
            HumanMessage(content=user_prompt)
        ]
        
        response = self.llm.invoke(messages)
        final_report = response.content
        
        # Update state
        state["final_report"] = final_report
        state["current_step"] = "complete"
        state["messages"].append({
            "role": "reporting",
            "content": "Final report compiled successfully",
            "report_length": len(final_report)
        })
        
        print(f"✓ Report compiled successfully")
        print(f"  Report length: {len(final_report)} characters")
        print(f"{'='*80}\n")
        
        return state
    
    def _format_research_data(self, questions: List[str], 
                             search_results: Dict[str, List[Dict]]) -> str:
        """
        Format research data for LLM processing.
        
        Args:
            questions: List of research questions
            search_results: Dictionary of search results per question
            
        Returns:
            Formatted string containing all research data
        """
        formatted_data = []
        
        for i, question in enumerate(questions, 1):
            formatted_data.append(f"\n### Question {i}: {question}\n")
            
            results = search_results.get(question, [])
            if results:
                formatted_data.append("**Sources:**\n")
                for j, result in enumerate(results[:5], 1):  # Limit to top 5 results
                    formatted_data.append(f"\n{j}. **{result['title']}**")
                    formatted_data.append(f"   URL: {result['url']}")
                    formatted_data.append(f"   Content: {result['content'][:500]}...")
                    formatted_data.append("")
            else:
                formatted_data.append("No search results found.\n")
        
        return "\n".join(formatted_data)
    
    # ========================================================================
    # GRAPH CONSTRUCTION
    # ========================================================================
    
    def _build_graph(self) -> StateGraph:
        """
        Build the LangGraph workflow for the ReAct agent.
        
        The graph follows a linear flow:
        START -> Planning -> Acting -> Reporting -> END
        
        Returns:
            Compiled StateGraph ready for execution
        """
        # Create the graph
        workflow = StateGraph(AgentState)
        
        # Add nodes
        workflow.add_node("planning", self.planning_node)
        workflow.add_node("acting", self.acting_node)
        workflow.add_node("reporting", self.reporting_node)
        
        # Define edges (flow)
        workflow.set_entry_point("planning")
        workflow.add_edge("planning", "acting")
        workflow.add_edge("acting", "reporting")
        workflow.add_edge("reporting", END)
        
        # Compile the graph
        return workflow.compile()
    
    # ========================================================================
    # MAIN EXECUTION METHOD
    # ========================================================================
    
    def research(self, topic: str) -> Dict:
        """
        Execute the research process for a given topic.
        
        This is the main entry point for using the agent.
        
        Args:
            topic: The research topic (e.g., "Climate Change", "Artificial Intelligence")
            
        Returns:
            Dictionary containing:
                - topic: The research topic
                - questions: Generated research questions
                - search_results: Raw search results
                - report: The final compiled report
                - timestamp: When the research was conducted
        """
        print(f"\n{'#'*80}")
        print(f"# WEB RESEARCH AGENT - ReAct Pattern")
        print(f"{'#'*80}")
        print(f"# Topic: {topic}")
        print(f"# Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"{'#'*80}\n")
        
        # Initialize state
        initial_state = {
            "topic": topic,
            "research_questions": [],
            "search_results": {},
            "final_report": "",
            "current_step": "initialized",
            "messages": []
        }
        
        # Execute the graph
        final_state = self.graph.invoke(initial_state)
        
        # Prepare output
        result = {
            "topic": topic,
            "questions": final_state["research_questions"],
            "search_results": final_state["search_results"],
            "report": final_state["final_report"],
            "timestamp": datetime.now().isoformat(),
            "messages": final_state["messages"],
        }
        
        print(f"\n{'#'*80}")
        print(f"# RESEARCH COMPLETE ✓")
        print(f"{'#'*80}\n")
        
        return result


# ============================================================================
# UTILITY FUNCTIONS
# ============================================================================

def display_report(result: Dict):
    """
    Display the research report in a formatted way.
    
    Args:
        result: The result dictionary from agent.research()
    """
    from IPython.display import Markdown, display
    
    print("\n" + "="*80)
    print("RESEARCH QUESTIONS")
    print("="*80)
    for i, q in enumerate(result['questions'], 1):
        print(f"{i}. {q}")
    
    print("\n" + "="*80)
    print("FINAL REPORT")
    print("="*80 + "\n")
    
    # Display report as markdown in notebook
    display(Markdown(result['report']))


def save_report(result: Dict, filename: str = "research_report.md"):
    """
    Save the research report to a file.
    
    Args:
        result: The result dictionary from agent.research()
        filename: Output filename (default: research_report.md)
    """
    with open(filename, 'w', encoding='utf-8') as f:
        f.write(f"# Research Report: {result['topic']}\n\n")
        f.write(f"**Generated:** {result['timestamp']}\n\n")
        f.write("## Research Questions\n\n")
        for i, q in enumerate(result['questions'], 1):
            f.write(f"{i}. {q}\n")
        f.write("\n---\n\n")
        f.write(result['report'])
    
    print(f"✓ Report saved to: {filename}")




In [ ]:
agent = WebResearchAgent(
    model_name="gpt-4o-mini",
    temperature=0.7,
    num_questions=5,
    max_search_results=5
)

result = agent.research("Artificial Intelligence in Healthcare")

In [ ]:
result

In [ ]:
display_report(result)